In [1]:
import torch
import numpy as np
from evaluation_metrics import get_classification_report, get_f1_score, get_auc_score, get_eer_score
import random
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor
from torchvision import transforms
from torch.utils.data import random_split
from MobileNet_model import get_trained_MobileNet_model
from ResNet50_model import get_trained_ResNet50_model
from VisionTransformer import get_trained_ViT_model

from import_data import get_sample_paths, PathLabelDataset, build_transform

/Users/majaambrozic/Documents/DRL/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1, Loss: 0.6664
Epoch 2, Loss: 0.5799
Epoch 3, Loss: 0.5176
Epoch 4, Loss: 0.4897
Epoch 5, Loss: 0.4524
Epoch 6, Loss: 0.4190
Epoch 7, Loss: 0.3714
Epoch 8, Loss: 0.3587
Epoch 9, Loss: 0.3647
Epoch 10, Loss: 0.3882
Epoch 1, Loss: 0.3164
Epoch 2, Loss: 0.2779
Epoch 3, Loss: 0.2929
Evaluating MobileNetV2...

MOBILENETV2 PERFORMANCE
F1 Score:  0.6537
AUC Score: 0.6890
EER Score: 0.3540

Classification Report:
               precision    recall  f1-score   support

           0     0.5741    0.7126    0.6359        87
           1     0.7283    0.5929    0.6537       113

    accuracy                         0.6450       200
   macro avg     0.6512    0.6528    0.6448       200
weighted avg     0.6612    0.6450    0.6459       200

Epoch 1, Loss: 0.6353
Epoch 2, Loss: 0.4814
Epoch 3, Loss: 0.3837
Epoch 4, Loss: 0.2921
Epoch 5, Loss: 0.2557
Unfreezing layer4
Unfreezing fc
Epoch 1, Loss: 0.1821
Epoch 2, Loss: 0.1273
Epoch 3, Loss: 0.1043
Evaluating ResNet50...

RESNET50 PERFORMANCE
F1 

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 9730.20it/s]
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Loss: 0.6929
Epoch 2 | Loss: 0.6304
Epoch 3 | Loss: 0.5524
Epoch 4 | Loss: 0.4484
Epoch 5 | Loss: 0.3307

VIT MODEL PERFORMANCE
F1 Score: 0.7421
AUC Score: 0.8184
EER Score: 0.2920

Classification Report:
              precision    recall  f1-score   support

           0     0.6630    0.7011    0.6816        87
           1     0.7593    0.7257    0.7421       113

    accuracy                         0.7150       200
   macro avg     0.7112    0.7134    0.7118       200
weighted avg     0.7174    0.7150    0.7158       200



In [2]:
random.seed(42)
N_SAMPLES_PER_CLASS = 500

def get_loaders(n=N_SAMPLES_PER_CLASS, batch_size=32, train_split=0.8):
    # 1. Collect and filter all sample paths
    paths, labels = get_sample_paths(n=n)
    
    # 2. Create the full dataset
    full_dataset = PathLabelDataset(paths, labels, transform=build_transform(224))

    # 3. Calculate lengths for split
    train_len = int(len(full_dataset) * train_split)
    test_len = len(full_dataset) - train_len

    # 4. Perform the random split
    train_ds, test_ds = random_split(
        full_dataset, 
        [train_len, test_len],
        generator=torch.Generator().manual_seed(42)
    )

    # 5. Create two separate loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

train_loader, test_loader = get_loaders()

In [3]:
# Get loaders for ViT
model_name = "google/vit-base-patch16-224-in21k" # pretrained model
processor = ViTImageProcessor.from_pretrained(model_name)

vit_transform = transforms.Compose([
    transforms.Resize((processor.size["height"], processor.size["width"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

def get_loaders_ViT(n=N_SAMPLES_PER_CLASS, batch_size=32, train_split=0.8):
    # 1. Collect and filter all sample paths
    paths, labels = get_sample_paths(n=n)
    
    # 2. Create the full dataset
    full_dataset = PathLabelDataset(paths, labels, transform=vit_transform)

    # 3. Calculate lengths for split
    train_len = int(len(full_dataset) * train_split)
    test_len = len(full_dataset) - train_len

    # 4. Perform the random split
    train_ds, test_ds = random_split(
        full_dataset, 
        [train_len, test_len],
        generator=torch.Generator().manual_seed(42)
    )

    # 5. Create two separate loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader

vit_train_loader, vit_test_loader = get_loaders_ViT()

In [4]:
def cnn_predict_fn(model, images):
    logits = model(images)
    return torch.sigmoid(logits).squeeze()

def vit_predict_fn(model, images):
    outputs = model(pixel_values=images)
    return torch.softmax(outputs.logits, dim=1)[:, 1]

In [7]:

class EnsembleVoter:
    def __init__(self, models: list[tuple[str, torch.nn.Module, callable, DataLoader]], device: str):
        self.models = models
        self.device = device

    def _predict_single_model(self, model, predict_fn, loader):
        """Run inference for one model, return (preds, probs)"""
        model.eval()
        all_preds = []
        all_probs = []

        with torch.no_grad():
            for images, _ in loader:
                images = images.to(self.device)
                probs = predict_fn(model, images).squeeze()
                preds = (probs > 0.5).float()

                if probs.dim() == 0:
                    all_probs.append(probs.item())
                    all_preds.append(preds.item())
                else:
                    all_probs.extend(probs.cpu().numpy())
                    all_preds.extend(preds.cpu().numpy())

        return np.array(all_preds), np.array(all_probs)

    def predict(self, mode="hard"):
        """
        mode: 
          "hard" → each model votes 0 or 1, majority wins
          "soft" → average the probabilities, then threshold
        
        Returns final_preds, all_model_preds (dict)
        """
        all_model_preds = {}
        all_model_probs = {}

        for name, model, predict_fn, loader in self.models:
            preds, probs = self._predict_single_model(model, predict_fn, loader)
            all_model_preds[name] = preds
            all_model_probs[name] = probs
            print(f"  [{name}] predictions collected")

        if mode == "hard":
            # Stack predictions → shape (n_models, n_samples)
            stacked = np.stack(list(all_model_preds.values()), axis=0)
            # Majority vote across models for each sample
            final_preds = (stacked.mean(axis=0) >= 0.5).astype(float)

        elif mode == "soft":
            # Average probabilities across models
            stacked_probs = np.stack(list(all_model_probs.values()), axis=0)
            avg_probs = stacked_probs.mean(axis=0)
            final_preds = (avg_probs >= 0.5).astype(float)

        return final_preds, all_model_preds

    def evaluate(self, mode="hard"):
        """Run ensemble prediction and print metrics"""
        all_labels = []
        _, _, _, loader = self.models[0]
        for _, labels in loader:
            all_labels.extend(labels.numpy())
        y_test = np.array(all_labels)

        print(f"\nRunning ensemble ({mode} voting)...")
        final_preds, per_model_preds = self.predict(mode=mode)

        # Per-model metrics
        print("\n--- Individual Model Performance ---")
        for name, preds in per_model_preds.items():
            f1 = get_f1_score(y_test, preds)
            print(f"  {name}: F1 = {f1:.4f}")

        # Ensemble metrics
        print("\n--- Ensemble Performance ---")
        print(f"F1 Score:  {get_f1_score(y_test, final_preds):.4f}")
        print(f"AUC Score: {get_auc_score(y_test, final_preds):.4f}")
        print(f"EER Score: {get_eer_score(y_test, final_preds):.4f}")
        print("\nClassification Report:\n", get_classification_report(y_test, final_preds))

In [ ]:
# After training your models...
device = "cuda" if torch.cuda.is_available() else "cpu"
ensemble = EnsembleVoter(
    models=[
        ("ResNet50", get_trained_ResNet50_model(train_loader, device), cnn_predict_fn, test_loader),
        ("MobileNetV2", get_trained_MobileNet_model(train_loader, device), cnn_predict_fn, test_loader),
        ("ViT", get_trained_ViT_model(vit_train_loader, device), vit_predict_fn, vit_test_loader)
    ],
    device=device
)

ensemble.evaluate(mode="hard")
ensemble.evaluate(mode="soft")

Epoch 1, Loss: 0.6361
Epoch 2, Loss: 0.5022
Epoch 3, Loss: 0.3879
Epoch 4, Loss: 0.3228
Epoch 5, Loss: 0.2507
Unfreezing layer4
Unfreezing fc
Epoch 1, Loss: 0.1842
Epoch 2, Loss: 0.1300
Epoch 3, Loss: 0.1094
Epoch 1, Loss: 0.6579
Epoch 2, Loss: 0.5715
Epoch 3, Loss: 0.5125
Epoch 4, Loss: 0.5109
Epoch 5, Loss: 0.4613
Epoch 6, Loss: 0.4286
Epoch 7, Loss: 0.4279
Epoch 8, Loss: 0.3820
Epoch 9, Loss: 0.3479
Epoch 10, Loss: 0.3429
Epoch 1, Loss: 0.2784
Epoch 2, Loss: 0.2655
Epoch 3, Loss: 0.2653


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 18974.85it/s]
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 | Loss: 0.6899
Epoch 2 | Loss: 0.6195
Epoch 3 | Loss: 0.5285
Epoch 4 | Loss: 0.4090
Epoch 5 | Loss: 0.2896

Running ensemble (hard voting)...
  [ResNet50] predictions collected
  [MobileNetV2] predictions collected
  [ViT] predictions collected

--- Individual Model Performance ---
  ResNet50: F1 = 0.7650
  MobileNetV2: F1 = 0.6540
  ViT: F1 = 0.7453

--- Ensemble Performance ---
F1 Score:  0.7570
AUC Score: 0.7435
EER Score: 0.2689

Classification Report:
               precision    recall  f1-score   support

           0     0.6768    0.7701    0.7204        87
           1     0.8020    0.7168    0.7570       113

    accuracy                         0.7400       200
   macro avg     0.7394    0.7435    0.7387       200
weighted avg     0.7475    0.7400    0.7411       200


Running ensemble (soft voting)...
  [ResNet50] predictions collected
  [MobileNetV2] predictions collected
  [ViT] predictions collected

--- Individual Model Performance ---
  ResNet50: F1 = 0.7650
  M